In [1]:
#code for running GravSphere2 from laptop (for parallelized version for multi-cpu machines see run_multcpu_example.py file)

#The gravsphere2 file defines the likelihood for sampling and opens the initialization file with the specific object
#with all the data, this is an example script of running the sampling routine + doing some plots

#Our default example is for the Fornax LOS + PM simulation mock with 100 tracers

from gravsphere2 import *

import matplotlib.pyplot as plt
    
import dynesty
from dynesty import plotting as dyplot
from dynesty import DynamicNestedSampler

%matplotlib inline

In [2]:

# Try to use an nlive parameter of the order of the dimensions squared (a few times ideally) and at least 50 points per mode
#the higher the nlive parameter the finer and more reliable the sampling (less likely you'll miss a narrow mode) but
#the higher the sampling / CPU time it will take
#Note: this will probably take about ~ 1 - 2 days in a normal laptop, if you want to run quickly use lower nlive, fix the photometry or (preferably)
#use a high-performance computing machine with more CPUs (see multi-CPU machine example)
#PMs + more tracers will make runs take longer

sampler = DynamicNestedSampler(lnprob, ptform, ndims, nlive = 500) #this creates new sampler


fsname = str(diro + 'Sampler Chains/fornax_pm100_example_chk')  #file for checkpointing 

#sampler = DynamicNestedSampler.restore(fsname) #this restores existing sampler to run it from where you left off

print('Running dynesty:')

#runs the sampler (this will run the sampler until maxiter number of steps have been reached, if use_stop = True the run 
#will stop when the sampler has converged
sampler.run_nested(checkpoint_file=fsname, use_stop = True,
                maxiter = 300000, print_progress=True)   

#sampler.run_nested(checkpoint_file=fsname, use_stop = True,
        #         maxiter = 300000, print_progress=True, resume = True) #if running from existing file with the .restore option

Running dynesty:


iter: 1254 | batch: 0 | bound: 0 | nc: 25 | ncall: 5626 | eff(%): 20.470 | loglstar: -32131.243 | logz: -32140.374 | dlogz: 2284.929 >  0.010                                                         

Exception while calling loglikelihood function:
  params: [-1.98140123e-01  2.56631842e-01 -1.52150116e+00  1.52353707e+00
  8.32532752e-01 -9.14839666e-01 -1.71790234e+00  2.37594962e+00
 -1.03485649e+00  2.05058613e+00  5.26670080e+00  1.11590094e+00
  7.88945337e+00  3.76178778e+01 -1.54707875e+00  3.08400134e-01
  1.16848485e+00  4.75523569e+00  4.48081422e+07]
  args: []
  kwargs: {}
  exception:


Traceback (most recent call last):
  File "/rds/general/user/jd925/home/.conda/envs/gravsphere/lib/python3.11/site-packages/dynesty/dynesty.py", line 803, in __call__
    return self.func(np.asarray(x).copy(), *self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/rds/general/user/jd925/home/PhD_first_year/GravSphere2/gravsphere2.py", line 219, in lnprob
    return lnlike(baranr,barsurf,M,beta,betaf, Mpars,\
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/rds/general/user/jd925/home/PhD_first_year/GravSphere2/gravsphere2.py", line 343, in lnlike_prop_k
    Sigout, pmt2, pmr2, los2, kt, kr, kl, neg = sigp_prop_k(R,rLOS,rPM,nu,Sigfunc,M,beta,betaf,Mpars,\
                                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/rds/general/user/jd925/home/PhD_first_year/GravSphere2/gs2_functions.py", line 513, in sigp_prop_k
    integrator(((\
  File "/rds/general/user/jd925/home/.con

KeyboardInterrupt: 

In [ ]:
#diagnostic plot of the run, if the sampler has converged, the number of live points and weights should go to zero

resu = sampler.results

print("number of sampling iterations = ", resu.niter)

from dynesty import plotting as dyplot

%matplotlib inline

# Plot a summary of the run.
rfig, raxes = dyplot.runplot(resu)

# Plot traces and 1-D marginalized posteriors.
tfig, taxes = dyplot.traceplot(resu)

# Plot the 2-D marginalized posteriors.
cfig, caxes = dyplot.cornerplot(resu
                              )
plt.show()

In [ ]:
# Note: for saving your results for the long run and avoid compatibility issues with 
# different dynesty / Python versions, it is recommended that you save your results
# directly (rather than the dynesty chain), this can be done as follows:

import pickle

#for saving the results:
with open("my_results", "wb") as fp:   #Pickling
    pickle.dump(resu, fp)

#for retrieving the results:
with open("my_results", "rb") as fp:   # Unpickling
    my_results = pickle.load(fp)